# Week 5b — Design trends analysis

Standard first pass on **`design_trends.csv`** from `design_trends.py`: one row per ranked Wikipedia article (totals, averages, momentum, LCD strings).

**Before you start:** in `week 5/week 5b/`, run `python design_trends.py` so **`design_trends.csv`** is current. (Long-format daily/monthly data is in **`design_trends_raw.csv`**, not used here.)

---

1. What does your dataset look like? `head()` / `info()`  
2. What is the distribution of the most important numeric column?  
3. Filter to a meaningful subset — what is in it?  
4. Group by a category and take the mean of a numeric column.  
5. Where are missing values? Are any columns incomplete?


## Setup

In [ ]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path(".")
TRENDS_CSV = DATA_DIR / "design_trends.csv"

assert TRENDS_CSV.is_file(), (
    f"Missing {TRENDS_CSV.resolve()}. Run: python design_trends.py"
)

print("pandas:", pd.__version__)

---
## 1. What does your dataset look like? `head()` and `info()`

Load **`design_trends.csv`**, peek at the first and last rows, then inspect dtypes and non-null counts.

In [ ]:
df = pd.read_csv(TRENDS_CSV, encoding="utf-8")
assert "total_views_in_window" in df.columns and "trend" in df.columns, (
    "Expected design_trends.csv from design_trends.py (summary by article)."
)

df.head(10)

In [ ]:
df.tail(8)

In [ ]:
df.info()

---
## 2. Distribution of the most important column

For **overall interest** across articles, the main numeric column is **`total_views_in_window`**. We use `describe()`, a **binned count table** (numpy only), and an optional **histogram** if `matplotlib` is installed (`python -m pip install matplotlib`).

In [ ]:
df["total_views_in_window"].describe()

In [ ]:
import numpy as np

values = df["total_views_in_window"]
counts, edges = np.histogram(values, bins=15)
binned = pd.DataFrame(
    {"bin_left": edges[:-1], "bin_right": edges[1:], "row_count": counts}
)
print("Binned distribution (works without matplotlib):")
binned

try:
    import matplotlib.pyplot as plt  # noqa: F401 — enables pandas .plot backend

    ax = values.plot.hist(
        bins=15,
        edgecolor="white",
        title="Total views in window (per article)",
    )
    ax.set_xlabel("total_views_in_window")
except ImportError:
    print(
        "\nMatplotlib is not installed; only the table above is shown. "
        "To add the chart: python -m pip install matplotlib"
    )

---
## 3. Filter to a meaningful subset — what is in it?

Example: articles with **`momentum == "Rising"`** (recent 30-day average at least 10% above the prior 30 days).

In [ ]:
rising = df[df["momentum"] == "Rising"].copy()

print("Rising articles:", len(rising), "of", len(df))
rising.sort_values("total_views_in_window", ascending=False)

---
## 4. Group by a category — mean of a numeric column

Group by **`momentum`** and take the mean of **`total_views_in_window`**.

In [ ]:
mean_views_by_momentum = (
    df.groupby("momentum", as_index=False)["total_views_in_window"]
    .mean()
    .rename(columns={"total_views_in_window": "mean_total_views_in_window"})
    .sort_values("mean_total_views_in_window", ascending=False)
)

mean_views_by_momentum

---
## 5. Missing values — which columns are incomplete?

`isna().sum()` counts nulls. Text columns may contain empty strings (not counted as NA by default).

In [ ]:
missing = df.isna().sum()
missing[missing > 0]

In [ ]:
df.isna().sum()

In [ ]:
# Include both legacy object and new string dtypes (pandas 3 compatible).
text_cols = df.select_dtypes(include=["object", "string"]).columns
empty_string_counts = {
    c: (df[c].astype(str).str.strip() == "").sum() for c in text_cols
}
pd.Series(empty_string_counts).rename("empty_or_whitespace_rows")

In [ ]:
print("All columns complete (no NA):", missing.sum() == 0)